In [1]:
import json
from pathlib import Path

BASE_DIR = Path("mock_data")

with open(BASE_DIR / "detections.json", encoding="utf-8") as f:
    detections = json.load(f)

with open(BASE_DIR / "image_meta.json", encoding="utf-8") as f:
    image_meta = json.load(f)

print("Detection image sayısı:", len(detections))
print("Metadata image sayısı:", len(image_meta))

Detection image sayısı: 40
Metadata image sayısı: 40


In [2]:
def bbox_center(det):
    cx = det["x"] + det["w"] / 2
    cy = det["y"] + det["h"] / 2
    return cx, cy

In [3]:
def pixel_to_gps(cx, cy, meta):
    width = meta["width_px"]
    height = meta["height_px"]

    corners = meta["corner_coordinates"]

    tl_lat, tl_lon = corners["top_left"]
    tr_lat, tr_lon = corners["top_right"]
    bl_lat, bl_lon = corners["bottom_left"]
    br_lat, br_lon = corners["bottom_right"]

    # normalize pixel position to 0..1
    u = cx / width
    v = cy / height

    # top edge interpolation
    top_lat = tl_lat * (1 - u) + tr_lat * u
    top_lon = tl_lon * (1 - u) + tr_lon * u

    # bottom edge interpolation
    bottom_lat = bl_lat * (1 - u) + br_lat * u
    bottom_lon = bl_lon * (1 - u) + br_lon * u

    # vertical interpolation
    lat = top_lat * (1 - v) + bottom_lat * v
    lon = top_lon * (1 - v) + bottom_lon * v

    return lat, lon

In [4]:
detections_with_gps = []

for image_entry in detections:
    image_id = image_entry["image_id"]

    meta = image_meta[image_id]

    enriched_detections = []

    for det in image_entry["detections"]:
        cx, cy = bbox_center(det)
        lat, lon = pixel_to_gps(cx, cy, meta)

        enriched = {
            **det,
            "center_pixel": [cx, cy],
            "lat": lat,
            "lon": lon
        }

        enriched_detections.append(enriched)

    detections_with_gps.append({
        "image_id": image_id,
        "capture_time": meta["capture_time"],
        "detections": enriched_detections
    })

In [5]:
detections_with_gps[0]

{'image_id': 'img_000860',
 'capture_time': '12:05',
 'detections': [{'class': 'truck',
   'confidence': 0.85,
   'x': 469.0,
   'y': 289.0,
   'w': 43.0,
   'h': 61.0,
   'center_pixel': [490.5, 319.5],
   'lat': 39.94980691166784,
   'lon': 32.854243953910846}]}